In [1]:
# general imports
import pandas as pd
import os
import json

In [2]:
# Optional environment setup.
# Set these before initializing vLLM, PyTorch, sentence-transformers, or transformers models.

# Restrict this notebook process to specific GPUs.
# Examples: "0" for one GPU, "0,1" for two GPUs.
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Optional: set for higher Hugging Face Hub rate limits and faster first-time downloads.
# os.environ["HF_TOKEN"] = "HF..."

# MMAI Examples

## 1. Trial summarization walkthrough


### Read in data

In [3]:
trials_path = "data/trials_for_summarize.csv"
trials = pd.read_csv(trials_path)
trials = trials.iloc[0:10,]

### Local mode

`summarize_trials(...)` uses the local in-memory vLLM backend by default. Run this cell if you want local mode.


### Local mode (vLLM) - not used in this local Windows run

vLLM is not installed in this environment (no Linux/CUDA vLLM wheel for this
machine), so the in-memory local backend is unavailable. Use the remote-mode
cell below, which sends the same prompts to a local
[llama.cpp](https://github.com/ggml-org/llama.cpp) server. The original
local-mode call was:

```python
from matchminer_ai.trials import summarize_trials

trial_results, metadata, qc_report = summarize_trials(
    trials, return_metadata=True, return_qc=True
)

# Optional: clear the cached local vLLM engine if GPU memory is tight.
# from matchminer_ai.llm.backends import clear_local_llm_cache
# clear_local_llm_cache()```


### Remote mode

Run this cell instead of the local-mode cell when you want summarization requests to go to an OpenAI-compatible chat-completions endpoint. The endpoint can be a hosted API, an already-running vLLM server, or a local vLLM server started with the optional helper below.


In [4]:
from matchminer_ai import load_config
from matchminer_ai.trials import summarize_trials

# Local run: every LLM step is served by llama.cpp instead of vLLM.
# Start the server first (Windows):
#   H:/Apps/llama.cpp/run_server_gemma4e2b.bat  ->  http://127.0.0.1:8081/v1
# config_local_llamacpp.yaml sets remote.enabled, the endpoint URL, the served
# model name ("gemma-4-E2B-it") and context/sampling sizes that fit one 16 GB GPU.
os.environ.setdefault("OPENAI_API_KEY", "not-needed")
config = load_config("config_local_llamacpp.yaml")

trial_results, metadata, qc_report = summarize_trials(
    trials, config=config, return_metadata=True, return_qc=True
)


H:\workspace\matchminer-ai-inference\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### View results

In [5]:
trial_results.head()

,trial_id,clinical_space_number,clinical_space_summary,general_exclusion_criteria,space_trial_id
0,NCT03319901,0,Age range allowed: 60 Years and older. Sex all...,"History of pneumonitis, heart failure, renal d...",NCT03319901-0
6,NCT04792489,0,Age range allowed: 18 years and older. Sex all...,"History of pneumonitis, heart failure, renal d...",NCT04792489-0
7,NCT04792489,1,Age range allowed: 18 years and older. Sex all...,"History of pneumonitis, heart failure, renal d...",NCT04792489-1
8,NCT04792489,2,Age range allowed: 18 years and older. Sex all...,"History of pneumonitis, heart failure, renal d...",NCT04792489-2
9,NCT04792489,3,Age range allowed: 18 years and older. Sex all...,"History of pneumonitis, heart failure, renal d...",NCT04792489-3


Metadata contains a snapshot of the config used and metadata on the model used. 

In [6]:
metadata

{'package': {'name': 'matchminer-ai', 'version': '0.4.0'},
 'config_snapshot': {'version': 0,
  'debug_mode': False,
  'model_metadata_cache_dir': '.mmai_cache/model_metadata',
  'remote': {'enabled': True,
   'server_urls': ['http://127.0.0.1:8081/v1'],
   'max_concurrent_requests': 4,
   'request_timeout': 1800,
   'max_retries': 3,
   'batch_size': 1000,
   'retry_backoff_base': 1.0},
  'trial': {'reasoning_parser': 'auto',
   'local': {'model_name': 'google/gemma-4-31B-it',
    'engine': {'max_model_len': 32768,
     'tensor_parallel_size': 1,
     'gpu_memory_utilization': 0.9,
     'language_model_only': True},
    'chat_template_kwargs': {'enable_thinking': True},
    'generation': {'temperature': 1.0,
     'top_p': 0.95,
     'top_k': 20,
     'min_p': 0.0,
     'presence_penalty': 1.5,
     'max_tokens': 20000,
     'repetition_penalty': 1.0,
     'skip_special_tokens': False}},
   'remote': {'model_name': 'gemma-4-E2B-it',
    'request_params': {'max_tokens': 6000,
     'temp

QC report on the trial summarization run

In [7]:
qc_report

,metric,value,denominator,percent,ids
0,trials_missing_in_output,0.0,10.0,0.000000,[]
1,trials_truncated_llm_response,0.0,10.0,0.000000,[]
2,trials_failed_inference,0.0,10.0,0.000000,[]
3,spaces_exceed_embedding_token_limit,0.0,24.0,0.000000,[]
4,spaces_per_trial_min,1.0,NaN,NaN,[]
5,spaces_per_trial_median,1.0,NaN,NaN,[]
6,spaces_per_trial_max,10.0,NaN,NaN,[]
7,trials_with_non_distinct_spaces,1.0,10.0,10.000000,[NCT06234423]
8,spaces_dropped_missing_keyword:Age,12.0,39.0,30.769231,"[NCT03319901-1, NCT03319901-2, NCT04644211-1, ..."
9,spaces_dropped_missing_keyword:Sex,12.0,39.0,30.769231,"[NCT03319901-1, NCT03319901-2, NCT04644211-1, ..."


Save checked-in trial example artifacts


In [8]:
trial_results.to_csv("output/trial_summaries.csv", index=None)
with open("output/trial_summarization_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
qc_report.to_csv("output/trial_qc_report.csv", index=None)

## 2. Patient summarization walkthrough


### Read in initial notes

In [9]:
notes_path = "data/note_set_1.csv"
notes = pd.read_csv(notes_path)

### Local mode

`summarize_patients(...)` uses the local in-memory vLLM backend by default. Run this cell if you want local mode.


In [10]:
from matchminer_ai.llm.backends import clear_local_llm_cache
clear_local_llm_cache()

### Local mode (vLLM) - not used in this local Windows run

vLLM is not installed in this environment (no Linux/CUDA vLLM wheel for this
machine), so the in-memory local backend is unavailable. Use the remote-mode
cell below, which sends the same prompts to a local
[llama.cpp](https://github.com/ggml-org/llama.cpp) server. The original
local-mode call was:

```python
from matchminer_ai.patients import summarize_patients

patient_summaries, metadata, qc_report = summarize_patients(
    notes, return_metadata=True, return_qc=True
)

# Optional: clear the cached local vLLM engine if GPU memory is tight.
# from matchminer_ai.llm.backends import clear_local_llm_cache
# clear_local_llm_cache()```


### Remote mode

Run this cell instead of the local-mode cell when you want summarization requests to go to an OpenAI-compatible chat-completions endpoint. The endpoint can be a hosted API, an already-running vLLM server, or a local vLLM server started with the optional helper below.


In [11]:
from matchminer_ai import load_config
from matchminer_ai.patients import summarize_patients

# Same llama.cpp endpoint as the trial step above.
os.environ.setdefault("OPENAI_API_KEY", "not-needed")
config = load_config("config_local_llamacpp.yaml")

patient_summaries, metadata, qc_report = summarize_patients(
    notes, config=config, return_metadata=True, return_qc=True
)


H:\workspace\matchminer-ai-inference\src\matchminer_ai\patients\prepare.py:31: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  normalized["note_date"] = pd.to_datetime(normalized["note_date"])


### Updating existing patient summaries

Run this optional cell after the local- or remote-mode cell above if you want to simulate a longitudinal update. It reads a second set of new notes, uses the existing `patient_summaries` as the starting point, then updates those summaries with the new notes.


In [12]:
from matchminer_ai.patients import summarize_patients

new_notes_path = "data/note_set_2.csv"
new_notes = pd.read_csv(new_notes_path)

existing_summaries = patient_summaries.assign(
    patient_summary=lambda df: (
        df["cancer_history_summary"].fillna("")
        + "\n\nBoilerplate:\n"
        + df["general_exclusion_criteria_evidence"].fillna("")
    )
)[["patient_id", "patient_summary"]]

patient_summaries, metadata, qc_report = summarize_patients(
    new_notes,
    config=config,
    existing_summaries=existing_summaries,
    return_metadata=True,
    return_qc=True,
)

### View results

In [13]:
patient_summaries.head()

,patient_id,last_note_date,cancer_history_summary,general_exclusion_criteria_evidence
0,11609,2026-05-03,Age: 71\nSex: Female\nCancer type: Glioblastom...,"History of essential hypertension, controlled ..."
1,12019,2026-05-14,Age: 52\nSex: Female\nCancer type: Invasive du...,History of hypertension controlled on lisinopr...
2,15687,2025-10-07,Age: 68\nSex: Male\nCancer type: Small cell lu...,History of hypertension controlled on lisinopr...
3,16916000000,2024-11-14,"Age: 21\nSex: Male\nCancer type: Osteosarcoma,...",None
4,1722,2025-02-10,Age: 64\nSex: Male\nCancer type: Gastric/Gastr...,"ECOG 1-2 (ambulatory, limited activity due to ..."


Metadata contains a snapshot of the config used and metadata on the models used. 

In [14]:
metadata

{'package': {'name': 'matchminer-ai', 'version': '0.4.0'},
 'config_snapshot': {'version': 0,
  'debug_mode': False,
  'model_metadata_cache_dir': '.mmai_cache/model_metadata',
  'remote': {'enabled': True,
   'server_urls': ['http://127.0.0.1:8081/v1'],
   'max_concurrent_requests': 4,
   'request_timeout': 1800,
   'max_retries': 3,
   'batch_size': 1000,
   'retry_backoff_base': 1.0},
  'trial': {'reasoning_parser': 'auto',
   'local': {'model_name': 'google/gemma-4-31B-it',
    'engine': {'max_model_len': 32768,
     'tensor_parallel_size': 1,
     'gpu_memory_utilization': 0.9,
     'language_model_only': True},
    'chat_template_kwargs': {'enable_thinking': True},
    'generation': {'temperature': 1.0,
     'top_p': 0.95,
     'top_k': 20,
     'min_p': 0.0,
     'presence_penalty': 1.5,
     'max_tokens': 20000,
     'repetition_penalty': 1.0,
     'skip_special_tokens': False}},
   'remote': {'model_name': 'gemma-4-E2B-it',
    'request_params': {'max_tokens': 6000,
     'temp

QC report on patient summarization

In [15]:
qc_report

,metric,value,denominator,percent,ids
0,patients_exceed_embedding_token_limit,0,12,0.0,[]
1,patients_exclusion_criteria_not_extracted,0,12,0.0,[]
2,patients_missing_keyword:Age,0,12,0.0,[]
3,patients_missing_keyword:Sex,0,12,0.0,[]
4,patients_missing_keyword:Cancer type,0,12,0.0,[]
5,patients_missing_keyword:Histology,0,12,0.0,[]
6,patients_missing_keyword:Current extent,0,12,0.0,[]
7,patients_missing_keyword:Biomarkers,0,12,0.0,[]
8,patients_missing_keyword:Treatment history,0,12,0.0,[]


Save checked-in patient example artifacts

In [16]:
patient_summaries.to_csv("output/patient_summaries.csv", index=None)
with open("output/patient_summarization_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
qc_report.to_csv("output/patient_qc_report.csv", index=None)

## 3. Embed summarized entities for matching


In [17]:
from matchminer_ai.embedding import embed_for_matching

In [18]:
# Uncomment if you want to start from previously generated summaries
# trial_results = pd.read_csv("output/trial_summaries.csv")
# patient_summaries = pd.read_csv("output/patient_summaries.csv", dtype="str")

### Trials

In [19]:
embedded_trials, metadata = embed_for_matching(
    trial_results, entity_type="trial", config=config, return_metadata=True
)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Loading weights:  11%|█         | 34/310 [00:00<00:00, 335.59it/s]

Loading weights:  22%|██▏       | 68/310 [00:00<00:00, 264.21it/s]

Loading weights:  31%|███       | 96/310 [00:00<00:00, 218.52it/s]

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 616.89it/s]

In [20]:
embedded_trials.head()

,space_trial_id,embedding
0,NCT03319901-0,"[-0.0177001953125, 0.0029754638671875, -0.0020..."
6,NCT04792489-0,"[-0.0260009765625, 0.01312255859375, 0.0040588..."
7,NCT04792489-1,"[-0.0247802734375, -0.0225830078125, 0.0043945..."
8,NCT04792489-2,"[-0.03955078125, 0.01519775390625, -0.00089645..."
9,NCT04792489-3,"[-0.058837890625, 0.0208740234375, 0.007873535..."


Metadata contains a snapshot of the config used and metadata on the models used. 

In [21]:
metadata

{'package': {'name': 'matchminer-ai', 'version': '0.4.0'},
 'config_snapshot': {'version': 0,
  'debug_mode': False,
  'model_metadata_cache_dir': '.mmai_cache/model_metadata',
  'remote': {'enabled': True,
   'server_urls': ['http://127.0.0.1:8081/v1'],
   'max_concurrent_requests': 4,
   'request_timeout': 1800,
   'max_retries': 3,
   'batch_size': 1000,
   'retry_backoff_base': 1.0},
  'trial': {'reasoning_parser': 'auto',
   'local': {'model_name': 'google/gemma-4-31B-it',
    'engine': {'max_model_len': 32768,
     'tensor_parallel_size': 1,
     'gpu_memory_utilization': 0.9,
     'language_model_only': True},
    'chat_template_kwargs': {'enable_thinking': True},
    'generation': {'temperature': 1.0,
     'top_p': 0.95,
     'top_k': 20,
     'min_p': 0.0,
     'presence_penalty': 1.5,
     'max_tokens': 20000,
     'repetition_penalty': 1.0,
     'skip_special_tokens': False}},
   'remote': {'model_name': 'gemma-4-E2B-it',
    'request_params': {'max_tokens': 6000,
     'temp

### Patients

In [22]:
embedded_patients, metadata = embed_for_matching(
    patient_summaries, entity_type="patient", config=config, return_metadata=True
)

In [23]:
embedded_patients.head()

,patient_id,embedding
0,11609,"[-0.0213623046875, 0.0, -0.0052490234375, -0.0..."
1,12019,"[-0.056396484375, 0.007049560546875, -0.010192..."
2,15687,"[0.0157470703125, -0.0089111328125, -0.0076904..."
3,16916000000,"[0.055419921875, -0.017822265625, -0.005432128..."
4,1722,"[0.0361328125, 0.01495361328125, -0.0073242187..."


Metadata contains a snapshot of the config used and metadata on the models used. 

In [24]:
metadata

{'package': {'name': 'matchminer-ai', 'version': '0.4.0'},
 'config_snapshot': {'version': 0,
  'debug_mode': False,
  'model_metadata_cache_dir': '.mmai_cache/model_metadata',
  'remote': {'enabled': True,
   'server_urls': ['http://127.0.0.1:8081/v1'],
   'max_concurrent_requests': 4,
   'request_timeout': 1800,
   'max_retries': 3,
   'batch_size': 1000,
   'retry_backoff_base': 1.0},
  'trial': {'reasoning_parser': 'auto',
   'local': {'model_name': 'google/gemma-4-31B-it',
    'engine': {'max_model_len': 32768,
     'tensor_parallel_size': 1,
     'gpu_memory_utilization': 0.9,
     'language_model_only': True},
    'chat_template_kwargs': {'enable_thinking': True},
    'generation': {'temperature': 1.0,
     'top_p': 0.95,
     'top_k': 20,
     'min_p': 0.0,
     'presence_penalty': 1.5,
     'max_tokens': 20000,
     'repetition_penalty': 1.0,
     'skip_special_tokens': False}},
   'remote': {'model_name': 'gemma-4-E2B-it',
    'request_params': {'max_tokens': 6000,
     'temp

In [25]:
# Optional local export if you want to reuse embeddings without rerunning this step.
embedded_patients.to_parquet("output/embedded_patients.parquet")
embedded_trials.to_parquet("output/embedded_trials.parquet")

## 4. Generate candidate matches

Generate ranked trial-space candidates for each patient.


In [26]:
from matchminer_ai.matching import generate_candidate_matches

In [27]:
# Uncomment if you saved embeddings locally and want to start from them
# embedded_patients = pd.read_parquet('output/embedded_patients.parquet')
# embedded_trials = pd.read_parquet('output/embedded_trials.parquet')

### Candidate matches


In [28]:
candidate_matches = generate_candidate_matches(
    query_df=embedded_patients, corpus_df=embedded_trials
)

In [29]:
candidate_matches.head()

,patient_id,space_trial_id,similarity_score,rank
0,11609,NCT05538130-0,0.407865,1
1,11609,NCT05785741-0,0.364100,2
2,11609,NCT06234423-0,0.284609,3
3,11609,NCT06234423-1,0.284609,4
4,11609,NCT06234423-2,0.284609,5


## 5. Match quality check


### Classifier-based match-quality check

Run the trained classifier checker after adding patient and trial summary context.

In [30]:
from matchminer_ai.matching import score_match_quality

Add back the patient and trial summary context required by the match-quality checker.


In [31]:
candidate_matches_with_context = (
    candidate_matches.merge(
        patient_summaries[["patient_id", "cancer_history_summary"]],
        on="patient_id",
        how="left",
    ).merge(
        trial_results[["space_trial_id", "clinical_space_summary"]],
        on="space_trial_id",
        how="left",
    )
)
candidate_matches_with_context.head()

,patient_id,space_trial_id,similarity_score,rank,cancer_history_summary,clinical_space_summary
0,11609,NCT05538130-0,0.407865,1,Age: 71\nSex: Female\nCancer type: Glioblastom...,Age range allowed: Not specified. Sex allowed:...
1,11609,NCT05785741-0,0.364100,2,Age: 71\nSex: Female\nCancer type: Glioblastom...,Age range allowed: ≥ 18 years of age or accept...
2,11609,NCT06234423-0,0.284609,3,Age: 71\nSex: Female\nCancer type: Glioblastom...,Age range allowed: ≥18 years. Sex allowed: Mal...
3,11609,NCT06234423-1,0.284609,4,Age: 71\nSex: Female\nCancer type: Glioblastom...,Age range allowed: ≥18 years. Sex allowed: Mal...
4,11609,NCT06234423-2,0.284609,5,Age: 71\nSex: Female\nCancer type: Glioblastom...,Age range allowed: ≥18 years. Sex allowed: Mal...


Run the match-quality checker.


In [32]:
match_quality_matches, metadata = score_match_quality(
    candidate_matches_with_context, config=config, return_metadata=True
)

H:\workspace\matchminer-ai-inference\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in H:\workspace\matchminer-ai-inference\examples\.mmai_cache\model_metadata\models--ksg-dfci--TrialChecker-0526. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/174 [00:00<?, ?it/s]

Loading weights:   4%|▍         | 7/174 [00:00<00:02, 66.00it/s]

Loading weights:   8%|▊         | 14/174 [00:00<00:02, 58.30it/s]

Loading weights:  11%|█▏        | 20/174 [00:00<00:04, 37.03it/s]

Loading weights:  14%|█▍        | 25/174 [00:00<00:03, 38.06it/s]

Loading weights: 100%|██████████| 174/174 [00:00<00:00, 282.84it/s]

In [33]:
match_quality_matches.head()

,patient_id,space_trial_id,match_quality_score,match_quality_pass
0,11609,NCT05538130-0,0.494348,True
1,11609,NCT05785741-0,0.539855,True
2,12019,NCT05785741-0,0.374146,True
3,15687,NCT05785741-0,0.532979,True
4,16916000000,NCT05785741-0,0.497023,True


In [34]:
metadata

{'package': {'name': 'matchminer-ai', 'version': '0.4.0'},
 'config_snapshot': {'version': 0,
  'debug_mode': False,
  'model_metadata_cache_dir': '.mmai_cache/model_metadata',
  'remote': {'enabled': True,
   'server_urls': ['http://127.0.0.1:8081/v1'],
   'max_concurrent_requests': 4,
   'request_timeout': 1800,
   'max_retries': 3,
   'batch_size': 1000,
   'retry_backoff_base': 1.0},
  'trial': {'reasoning_parser': 'auto',
   'local': {'model_name': 'google/gemma-4-31B-it',
    'engine': {'max_model_len': 32768,
     'tensor_parallel_size': 1,
     'gpu_memory_utilization': 0.9,
     'language_model_only': True},
    'chat_template_kwargs': {'enable_thinking': True},
    'generation': {'temperature': 1.0,
     'top_p': 0.95,
     'top_k': 20,
     'min_p': 0.0,
     'presence_penalty': 1.5,
     'max_tokens': 20000,
     'repetition_penalty': 1.0,
     'skip_special_tokens': False}},
   'remote': {'model_name': 'gemma-4-E2B-it',
    'request_params': {'max_tokens': 6000,
     'temp

### LLM-based match-quality check

This optional checker uses the configured LLM backend instead of the classifier checker. Debug mode includes raw response columns for reviewing parsed scores; separate reasoning columns are populated only when the backend returns reasoning content.

In [35]:
from matchminer_ai.matching import score_match_quality_with_llm

# The LLM checker is much slower than the classifier checker, so on a single
# local GPU we only run it on the first 10 candidate pairs.
llm_match_quality_matches, llm_match_quality_metadata = score_match_quality_with_llm(
    candidate_matches_with_context.head(10),
    config=config,
    return_metadata=True,
)


In [36]:
llm_match_quality_matches.head()

,patient_id,space_trial_id,llm_match_quality_score
0,11609,NCT05538130-0,3
1,11609,NCT05785741-0,5
2,11609,NCT06234423-0,3
3,11609,NCT06234423-1,4
4,11609,NCT06234423-2,3


## 6. Exclusion criteria check


### Classifier-based exclusion criteria check

Run the trained classifier checker after adding patient and trial exclusion-criteria context.

In [37]:
# limit to unique patient-trial ID pairs
match_quality_matches["trial_id"] = (
    match_quality_matches["space_trial_id"].str.split("-").str[0]
)
unique_patient_trial_pairs = match_quality_matches[
    ["patient_id", "trial_id"]
].drop_duplicates()

# add in exclusion criteria for patients and trials
patient_trial_pairs_with_exclusion_context = unique_patient_trial_pairs.merge(
    patient_summaries[["patient_id", "general_exclusion_criteria_evidence"]],
    on="patient_id",
    how="left",
).merge(
    trial_results[["trial_id", "general_exclusion_criteria"]].drop_duplicates(),
    on="trial_id",
    how="left",
)
patient_trial_pairs_with_exclusion_context.head()

,patient_id,trial_id,general_exclusion_criteria_evidence,general_exclusion_criteria
0,11609,NCT05538130,"History of essential hypertension, controlled ...",History or current evidence of retinal vein oc...
1,11609,NCT05785741,"History of essential hypertension, controlled ...",History of symptomatic congestive heart failur...
2,12019,NCT05785741,History of hypertension controlled on lisinopr...,History of symptomatic congestive heart failur...
3,15687,NCT05785741,History of hypertension controlled on lisinopr...,History of symptomatic congestive heart failur...
4,16916000000,NCT05785741,None,History of symptomatic congestive heart failur...


In [38]:
from matchminer_ai.matching import exclusion_criteria_check

# run exclusion criteria check
exclusion_results, metadata = exclusion_criteria_check(
    patient_trial_pairs_with_exclusion_context, config=config, return_metadata=True
)
exclusion_results.head()

H:\workspace\matchminer-ai-inference\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in H:\workspace\matchminer-ai-inference\examples\.mmai_cache\model_metadata\models--ksg-dfci--BoilerplateChecker-0526. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/174 [00:00<?, ?it/s]

Loading weights:   8%|▊         | 14/174 [00:00<00:01, 133.67it/s]

Loading weights:  16%|█▌        | 28/174 [00:00<00:01, 99.44it/s] 

Loading weights:  23%|██▎       | 40/174 [00:00<00:01, 103.14it/s]

Loading weights:  29%|██▉       | 51/174 [00:00<00:01, 80.42it/s] 

Loading weights: 100%|██████████| 174/174 [00:00<00:00, 269.50it/s]

,patient_id,trial_id,exclusion_score,exclusion_criteria_pass
0,11609,NCT05538130,0.999940,True
1,11609,NCT05785741,0.999653,True
2,12019,NCT05785741,0.999997,True
3,15687,NCT05785741,0.999869,True
4,16916000000,NCT05785741,1.000000,True


In [39]:
exclusion_results["exclusion_criteria_pass"].value_counts()

exclusion_criteria_pass
True    16
Name: count, dtype: int64

In [40]:
metadata

{'package': {'name': 'matchminer-ai', 'version': '0.4.0'},
 'config_snapshot': {'version': 0,
  'debug_mode': False,
  'model_metadata_cache_dir': '.mmai_cache/model_metadata',
  'remote': {'enabled': True,
   'server_urls': ['http://127.0.0.1:8081/v1'],
   'max_concurrent_requests': 4,
   'request_timeout': 1800,
   'max_retries': 3,
   'batch_size': 1000,
   'retry_backoff_base': 1.0},
  'trial': {'reasoning_parser': 'auto',
   'local': {'model_name': 'google/gemma-4-31B-it',
    'engine': {'max_model_len': 32768,
     'tensor_parallel_size': 1,
     'gpu_memory_utilization': 0.9,
     'language_model_only': True},
    'chat_template_kwargs': {'enable_thinking': True},
    'generation': {'temperature': 1.0,
     'top_p': 0.95,
     'top_k': 20,
     'min_p': 0.0,
     'presence_penalty': 1.5,
     'max_tokens': 20000,
     'repetition_penalty': 1.0,
     'skip_special_tokens': False}},
   'remote': {'model_name': 'gemma-4-E2B-it',
    'request_params': {'max_tokens': 6000,
     'temp

### LLM-based exclusion criteria check

This optional checker uses the configured LLM backend instead of the classifier checker and returns whether each patient-trial pair passes exclusion criteria.

In [41]:
from matchminer_ai.matching import exclusion_criteria_check_with_llm

# Same as above: a small sample keeps this optional step to a few minutes locally.
llm_exclusion_results, llm_exclusion_metadata = exclusion_criteria_check_with_llm(
    patient_trial_pairs_with_exclusion_context.head(10),
    config=config,
    return_metadata=True,
)


In [42]:
llm_exclusion_results.head()

,patient_id,trial_id,llm_exclusion_criteria_pass
0,11609,NCT05538130,True
1,11609,NCT05785741,True
2,12019,NCT05785741,True
3,15687,NCT05785741,True
4,16916000000,NCT05785741,True


## 7. Save final matches

Create one review table with one row per patient/trial pair. Each pair is represented by its top-scoring clinical space, rows are ranked by that top match-quality score, and exclusion-check results are included as labels without filtering rows.


In [43]:
patient_context_cols = [
    col
    for col in [
        "patient_id",
        "cancer_history_summary",
        "general_exclusion_criteria_evidence",
    ]
    if col in patient_summaries.columns
]
trial_context_cols = [
    col
    for col in [
        "space_trial_id",
        "clinical_space_number",
        "clinical_space_summary",
        "general_exclusion_criteria",
    ]
    if col in trial_results.columns
]
original_trial_cols = [
    col
    for col in [
        "trial_id",
        "oncore_id",
        "trial_title",
        "brief_summary",
        "detailed_summary",
        "eligibility_criteria",
    ]
    if col in trials.columns
]

final_matches = (
    match_quality_matches.merge(
        candidate_matches[["patient_id", "space_trial_id", "similarity_score", "rank"]],
        on=["patient_id", "space_trial_id"],
        how="left",
    )
    .merge(
        exclusion_results,
        on=["patient_id", "trial_id"],
        how="left",
    )
    .merge(
        patient_summaries[patient_context_cols],
        on="patient_id",
        how="left",
    )
    .merge(
        trial_results[trial_context_cols].drop_duplicates("space_trial_id"),
        on="space_trial_id",
        how="left",
    )
    .merge(
        trials[original_trial_cols].drop_duplicates("trial_id"),
        on="trial_id",
        how="left",
    )
)

# Keep one row per patient/trial pair, represented by the top-scoring clinical space.
final_matches = (
    final_matches.sort_values(
        ["patient_id", "trial_id", "match_quality_score"],
        ascending=[True, True, False],
    )
    .drop_duplicates(["patient_id", "trial_id"], keep="first")
    .sort_values(["patient_id", "match_quality_score"], ascending=[True, False])
    .reset_index(drop=True)
)
final_matches["match_quality_rank"] = final_matches.groupby("patient_id").cumcount() + 1

final_matches = final_matches.rename(
    columns={
        "match_quality_rank": "match_rank",
        "exclusion_criteria_pass": "passes_exclusion_criteria",
        "cancer_history_summary": "patient_summary",
        "general_exclusion_criteria_evidence": "patient_exclusion_evidence",
        "brief_summary": "trial_brief_summary",
        "detailed_summary": "trial_detailed_summary",
        "eligibility_criteria": "trial_eligibility_criteria",
        "clinical_space_summary": "matched_clinical_space_summary",
        "general_exclusion_criteria": "extracted_trial_exclusion_criteria",
    }
)

ordered_cols = [
    "patient_id",
    "trial_id",
    "match_rank",
    "passes_exclusion_criteria",
    "patient_summary",
    "patient_exclusion_evidence",
    "trial_title",
    "trial_brief_summary",
    "trial_eligibility_criteria",
    "clinical_space_number",
    "matched_clinical_space_summary",
    "extracted_trial_exclusion_criteria",
]
final_matches = final_matches[[col for col in ordered_cols if col in final_matches.columns]]
final_matches.head()


,patient_id,trial_id,match_rank,passes_exclusion_criteria,patient_summary,patient_exclusion_evidence,trial_title,trial_brief_summary,trial_eligibility_criteria,clinical_space_number,matched_clinical_space_summary,extracted_trial_exclusion_criteria
0,11609,NCT05785741,1,True,Age: 71\nSex: Female\nCancer type: Glioblastom...,"History of essential hypertension, controlled ...","A Phase 1/2a, Multicenter, Open-Label, First i...",This is a dose-escalation and dose-expansion P...,Inclusion Criteria:\n\n1. Male or female adult...,0,Age range allowed: ≥ 18 years of age or accept...,History of symptomatic congestive heart failur...
1,11609,NCT05538130,2,True,Age: 71\nSex: Female\nCancer type: Glioblastom...,"History of essential hypertension, controlled ...",A PHASE 1A/B OPEN-LABEL MASTER STUDY OF PF-077...,The purpose of this clinical trial is to learn...,Inclusion Criteria:\n\n* Diagnosis of advanced...,0,Age range allowed: Not specified. Sex allowed:...,History or current evidence of retinal vein oc...
2,12019,NCT05785741,1,True,Age: 52\nSex: Female\nCancer type: Invasive du...,History of hypertension controlled on lisinopr...,"A Phase 1/2a, Multicenter, Open-Label, First i...",This is a dose-escalation and dose-expansion P...,Inclusion Criteria:\n\n1. Male or female adult...,0,Age range allowed: ≥ 18 years of age or accept...,History of symptomatic congestive heart failur...
3,15687,NCT05785741,1,True,Age: 68\nSex: Male\nCancer type: Small cell lu...,History of hypertension controlled on lisinopr...,"A Phase 1/2a, Multicenter, Open-Label, First i...",This is a dose-escalation and dose-expansion P...,Inclusion Criteria:\n\n1. Male or female adult...,0,Age range allowed: ≥ 18 years of age or accept...,History of symptomatic congestive heart failur...
4,16916000000,NCT05785741,1,True,"Age: 21\nSex: Male\nCancer type: Osteosarcoma,...",None,"A Phase 1/2a, Multicenter, Open-Label, First i...",This is a dose-escalation and dose-expansion P...,Inclusion Criteria:\n\n1. Male or female adult...,0,Age range allowed: ≥ 18 years of age or accept...,History of symptomatic congestive heart failur...


In [44]:
# Optional local export for review or downstream analysis.
final_matches.to_csv("output/final_matches.csv", index=None)